In [8]:
import os

from SIMULATION_library import simulator_utils
from SIMULATION_library.fch_setup import simulation
from SIMULATION_library.cell_sims_utils import generate_bench_script

from GSA_library import ionic_output
from GSA_library.plotting import plot_Land_output

In [9]:
# sims_number = 100
# Probably won't change

username                        = "crg17"

# Particular experiment

folder_experiment_name = "rodero_healthy/h01/peri_0.9_kperi_0.005_aatria_3_treflv_100_trefrv_100_trefa_50"
num_beats_limit_cycle  = 500
num_beats              = 5
bpm                    = 60
bcl                    = int(60000/bpm)

# ARCHER2
states_folder = f"/work/e348/e348/{username}/rodero_healthy/states/" # put here the .sv files on the hpc

# Local computer
base_folder_local   = f"/media/croderog/SeagateExpansionDrive/{folder_experiment_name}"
cell_sims_basefolder = f"{base_folder_local}/cell_sims"
torord_folder       = f"{base_folder_local}/SS/param"
courtemanche_folder = f"{base_folder_local}/SS/param"
data_folder          = f"{base_folder_local}/data"
json_folder = f"{base_folder_local}/json_files"

os.makedirs(cell_sims_basefolder, exist_ok=True)

# First we generate the simulation scripts. That will generate auxiliary cell simulation files

In [10]:

# X = np.loadtxt(os.path.join(base_folder_local,'data/X.txt'),dtype=float)
N = 2

# -------------------------------------------------------------------------
# write simulation scripts

cmd = f"python write_simulation_scripts.py"
cmd += f" --datafolder {data_folder}"
cmd += " --fields mechanics"
cmd += " --platform archer2"
cmd += " --user crg17"
cmd += f" --paramfolder {base_folder_local}/json_files/"
cmd += f" --paramfolder_cell {base_folder_local}/SS/param/"
cmd += f" --slrmfolder {base_folder_local}/slrm/"
cmd += f" --HPC_statefolder {states_folder}"
cmd += f" --defaultfile {base_folder_local}/json_files/default.json"
cmd += " --idx1 0"
cmd += f" --idx2 {N-1}"
cmd += f" --clinical_data {base_folder_local}/json_files/clinical_data.json"
cmd += f" --tags {base_folder_local}/json_files/tags.json"
cmd += f" --setup_file {base_folder_local}/json_files/settings.json"
os.system(cmd)


Loading /media/croderog/SeagateExpansionDrive/rodero_healthy/h01/peri_0.9_kperi_0.005_aatria_3_treflv_100_trefrv_100_trefa_50/json_files/settings.json...
Done.
generating json file...
mechanics
(100, 6)
#######################################
    visualising setup for simulation   
#######################################
----------------------------
Four-chamber clock:
BCL : 1000.0
SA_t0 : -100.0
AV_delay : 100.0
LA_LV_delay : 100.0
RA_RV_delay : 100.0
ERP : 800.0
SA_RA_delay : 0.0
AA_delay : 0.0
VV_delay : 0.0
RA_AVa_delay : 0.0
----------------------------
----------------------------
Stimulus @ SAN:
stimID : 0
name : SAN
vtx_file : /work/e348/e348/crg17/meshes/h01//SAN
BCL : 1000.0
npls : 5
xtrg : 0
stim_type : 0
start : 0.0
strength : 60.0
duration : 2.0
xtrg_offset : 0.0
----------------------------
----------------------------
Stimulus @ fascicles_rv:
stimID : 1
name : fascicles_rv
vtx_file : /work/e348/e348/crg17/meshes/h01//fascicles_rv
BCL : 1000.0
npls : 5
xtrg : 1
stim_type 

mkdir: cannot create directory ‘/media/croderog/SeagateExpansionDrive/rodero_healthy/h01/peri_0.9_kperi_0.005_aatria_3_treflv_100_trefrv_100_trefa_50/json_files/’: File exists


----------------------------
Ionic model @ lv:
impID : 0
name : lv
ionic_model : ToRORd_dynCl
IDs_list : [1, 25, 29]
param : flags=ENDO,GNa=11.7802,GNaL_b=0.0279,Gto_b=0.16,PCa_b=8.3757e-05,GKr_b=0.0321,GKs_b=0.0011,GK1_b=0.6992,Gncx_b=0.0034,Pnak_b=15.4509,Jrel_b=1.5378,Jup_b=1.0,GpCa=0.0005,GKb_b=0.0189,PNab=1.9239e-09,PCab=5.9194e-08,GClCa=0.2843,GClb=0.00198,INaCa_fractionSS=0.35,ICaL_fractionSS=0.8,aCaMK=0.05,bCaMK=0.00068,CaMKo=0.05,cmdnmax_b=0.05,trpnmax=0.07,BSRmax=0.047,BSLmax=1.124,csqnmax=10.0,tauCa=0.2,tauTr=60.0
plugin : LandHumanStress
plug_param : Tref=100.0,perm50=0.35,nperm=2.036,TRPN_n=2.0,koff=0.1,dr=0.25,wfrac=0.5,TOT_A=25.0,ktm_unblock=0.021,beta_1=-2.4,beta_0=2.3,gamma=0.0085,gamma_wu=0.615,phi=2.23,ca50=0.805,nu=7.0,mu=3.0
im_sv_init : /work/e348/e348/crg17/rodero_healthy/states//ToRORd_dynCl_LandHumanStress.sv
----------------------------
----------------------------
Ionic model @ rv:
impID : 1
name : rv
ionic_model : ToRORd_dynCl
IDs_list : [2, 28]
param : flag

0

# Ventricles: ToRORd_dynCl + LandHumanStress

In [12]:
sim_setup = simulation()
sim_setup.load(f"{json_folder}/settings.json")


generate_bench_script(N=N,								# how many simulations to run (one per param.json)
					  BCL=bcl,						    	# BCL
					  NBEATS=num_beats,							# NBEATS
					  basefolder=f"{base_folder_local}/SS/",				# basefolder containing the param folder
					  NPROC=23,							# number of CPUs to use for parallel runs
					  strain=0.0,								# strain
					  chamber="LV",		 						# chamber = LV, RV or atria
					  contraction_model=sim_setup.contraction_model, 		# contraction model
					  suffix="")

os.system(f"bash {base_folder_local}/SS/run_ToRORd_dynCl.sh")


Loading /media/croderog/SeagateExpansionDrive/rodero_healthy/h01/peri_0.9_kperi_0.005_aatria_3_treflv_100_trefrv_100_trefa_50/json_files/settings.json...
Done.


mkdir: cannot create directory ‘/media/croderog/SeagateExpansionDrive/rodero_healthy/h01/peri_0.9_kperi_0.005_aatria_3_treflv_100_trefrv_100_trefa_50/SS/ToRORd_dynCl/’: File exists


0

# Atria: Converted COURTEMANCHE + LandHumanStress

In [13]:
sim_setup = simulation()
sim_setup.load(f"{json_folder}/settings.json")


generate_bench_script(N=N,								# how many simulations to run (one per param.json)
					  BCL=bcl,						    	# BCL
					  NBEATS=num_beats,							# NBEATS
					  basefolder=f"{base_folder_local}/SS/",				# basefolder containing the param folder
					  NPROC=23,							# number of CPUs to use for parallel runs
					  strain=0.0,								# strain
					  chamber="atria",		 						# chamber = LV, RV or atria
					  contraction_model=sim_setup.contraction_model, 		# contraction model
					  suffix="")

os.system(f"bash {base_folder_local}/SS/run_converted_COURTEMANCHE.sh")


Loading /media/croderog/SeagateExpansionDrive/rodero_healthy/h01/peri_0.9_kperi_0.005_aatria_3_treflv_100_trefrv_100_trefa_50/json_files/settings.json...
Done.


0